## RAG

In [ ]:
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS
from langchain.chains import RetrievalQA
from langchain.llms import HuggingFacePipeline
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM, pipeline

# === Step 1: Prepare the Dataset ===
print("Preparing the dataset...")
# A small dataset of text chunks
texts = [
    "The Eiffel Tower is located in Paris, France.",
    "The Great Wall of China is one of the Seven Wonders of the World.",
    "The Statue of Liberty is a symbol of freedom in the United States.",
    "Mount Everest is the highest mountain in the world.",
    "The Amazon Rainforest produces about 20% of Earth's oxygen."
]

# === Step 2: Create Embeddings and Vector Store ===
print("Creating embeddings and vector store...")
# Use Sentence-Transformers for embeddings
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

# Create a FAISS vector store from the texts
vector_store = FAISS.from_texts(texts, embeddings)


In [ ]:
# === Step 3: Load the Language Model ===
print("Loading the language model...")
# Use a pre-trained T5 model for text generation
model_name = "google/flan-t5-small"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForSeq2SeqLM.from_pretrained(model_name)

# Create a HuggingFace pipeline for text generation
text_generator = pipeline("text2text-generation", model=model, tokenizer=tokenizer)

# Wrap the pipeline in LangChain's HuggingFacePipeline
llm = HuggingFacePipeline(pipeline=text_generator)

In [ ]:
# === Step 4: Create the RAG Chain ===
print("Creating the RAG chain...")
# Combine the retriever and the language model into a RetrievalQA chain
retriever = vector_store.as_retriever()
qa_chain = RetrievalQA.from_chain_type(
    llm=llm,
    chain_type="stuff",
    retriever=retriever,
    return_source_documents=True
)

In [ ]:
# === Step 5: Query the RAG Model ===
print("Querying the RAG model...")
query = "Where is the Eiffel Tower located?"
result = qa_chain({"query": query})

# Print the result
print(f"Question: {query}")
print(f"Answer: {result['result']}")
print("Source Documents:")
for doc in result["source_documents"]:
    print(f"- {doc.page_content}")